# **PPG Preprocessing**

Pipeline for preprocessing raw PPG signal, developed as part of the Brain-Body Analysis Special Interest Group (BBSIG). 

To know how to use this PPG Preprocessing notebook, visit our step-by-step tutorial: https://martager.github.io/bbsig/ppg-preprocessing/

Jupyter notebook created by Elias Reinwarth, Agata Patyczek, Anthony Ciston, Niket Agrawal, & Marta Gerosa

Created on: 15 November 2024

Last update (by M. Gerosa): 11 April 2025

If you use this BBSIG pipeline in a publication, please cite us: *Gerosa M., Agrawal N., Ciston A.B., Fischer A., Fourcade A., Koushik A., Neubauer M., Patyczek A., Piejka A., Reinwarth E., Roellecke L., Shum Y.H., Verschooren S., Gaebler M. (2025). Brain-Body Analysis Special Interest Group (BBSIG) (Version 0.0.1) [Computer software]. https://martager.github.io/bbsig/*

## **Pipeline structure**

The following steps are included:

1. **Data import and conversion**: import the BIDS-compliant `_physio.tsv.gz` and `_physio.json` files containing the raw PPG signal and its metadata, then convert them into appropriate formats for later processing stages. 

2. **(Optional) PPG normalization, filtering & clipping artifacts correction**: normalize the signal between 1 and -1 (Sect. 2a), clean the PPG signal using NeuroKit2’s `signal_clean()` function (0.5 Hz high-pass and 8Hz low-pass 3rd order Butterworth filters; see Sect. 2b). Automatically handles clipping artifacts with Systole’s `find_clipping()` to identify the clipping threshold and `interpolate_clipping()` to interpolate the missing clipped peaks (or troughs); see Sect. 2c. 

3. **PPG peak detection**: custom function to detect PPG systolic peaks using NeuroKit2’s `nk.ppg_peaks()` with the 'elgendi' method. Optionally, two complementary automated artifact correction methods can be selected (i.e. NeuroKit2’s internal correction and/or Systole's `correct_peaks()`). Saves uncorrected (and automatically corrected) peak indices, as well as metadata about corrected artifacts (if applicable).

4. **(Optional) interactive visualization**: plot an interactive visualization of ECG signal with systolic peaks and instantaneous heart rate using Systole’s `plot_raw()` function, or produce interactive sub-space plots to identify artifacts (ectopic beats, long/short intervals) using Systole’s `plot_subspaces()` function. Plots can either be shown inside the notebook (if `plot_within_notebook` is set to `True`) or opened in a browser as HTML files. 

5.  **Manual peak correction**: manually identify and correct mis-detected PPG peaks and label bad segments using Systole’s `Editor`(saves output to `manual-correction.json` in the `derivatives` folder). It is recommended to run this section one participant at a time. 

6. **Data output**: export the `_ppg-cleaned.tsv.gz` (optional) and `_ppg-preproc.json` files for each subject in BIDS-compliant format to `/derivatives/ecg-preproc/sub-xx/`. Additionally, an `_hr-bpm-{correction_type}.tsv.gz` file can be saved with interpolated HR (in bpm).

In [ ]:
############## Import modules ##############

import numpy as np
import pandas as pd
import os, json
import neurokit2 as nk
from systole.detection import rr_artefacts
from systole.utils import input_conversion, heart_rate
from systole.plots import plot_raw, plot_subspaces
from systole.interact import Editor
from systole.detection import find_clipping, interpolate_clipping
from systole.correction import correct_peaks
from IPython.display import display
from bokeh.plotting import show, output_notebook

## **Settings: optional pipeline steps**

This section defines a series of variables that can be set to `True` to include the corresponding pipeline steps:

| Variable name | Function |
| --- | --- |
| **`ppg_normalize`** (bool) | **PPG normalisation** (sect. 2a): normalizes the raw PPG signal between 1 and -1. |
| **`ppg_filter`** (bool) | **PPG filtering** (sect. 2b): cleans the PPG signal by applying 0.5-8 Hz band-pass 3rd order Butterworth filters. |
| **`clip_artifacts_correct`** (bool) | **PPG clipping artifacts detection and interpolation** (sect. 2c): performs automated clipping artifact correction using Systole's `find_clipping()` to identify the clipping threshold and `interpolate_clipping()` to interpolate the missing clipped peaks (or troughs). |
| **`correct_artifacts_nk`** (bool) | **PPG automated artifact correction by NeuroKit2** (sect. 3): performs automated artifact correction of the PPG systolic peaks using NeuroKit2's `ppg_peaks()` function.| 
| **`correct_artifacts_sys`** (bool) <br>**`iterations = 1`** (numeric) | **PPG automated artifact correction by Systole** (sect. 3):  performs automated artifact correction of the PPG systolic peaks using Systole's `correct_peaks()` and stores details regarding each type of artifact from RR intervals ("ectopic", "short", "long", "missed", "extra"). The number of detection-correction iterations can be set using the `iterations` variable (defaults to `1`). |
| **`interactive_ppg_plot`** (bool) <br>**`participant_plots = []`** (list) | **Interactive plots of PPG signal with peaks or sub-spaces with artifacts** (sect. 4): displays an interactive visualization of the continuous PPG signal with systolic peaks and instantaneous HR, as well as interactive sub-spaces plots to identify artifacts. The participant IDs to-be-plotted can be specified in the list `participant_plots = []`; if left empty, all participants plots will be shown. |
| **`manual_correct`** (bool) <br>**`participant_manual = []`** (list) | **(Optional) manual correction of PPG systolic peaks** (sect. 5): activates UI for manual correction of extra peaks, missed peaks and/or falsely detected peaks, using Systole's `Editor`. The UI can also be used to annotate bad segments. Saves a JSON file with the corrected peaks and bad segments. It is recommended to run this section one participant at a time, by specifying the desired ID under `participant_manual = []` each time. | 
| **`hr_interpol`** (bool) | **Save HR interpolation** (sect. 6): exports interpolated heart rate (HR) values in BPM, using Systole's `utils.heart_rate()`, as TSV file with same sampling rate as original recording. |

In [ ]:
############## Settings: optional pipeline steps ##############

# For S2a: set whether (optional) PPG signal normalization is needed
ppg_normalize = True

# For S2b: set whether (optional) PPG signal filtering is needed
ppg_filter = True

# For S2c: set whether (optional) artifact clipping is needed
clip_artifacts_correct = True

# For S3: set whether (optional) PPG peaks artifacts correction using the in-built NeuroKit2 method
# based on Lipponen & Tarvainen (2019) is needed
correct_artifacts_nk = True

# For S3: set whether (optional) PPG peaks correction using Systole's correct_peaks() function is needed,
# and how many iterations to repeat the detection-correction process (default to 1)
correct_artifacts_sys = True
iterations = 1

# For S4: set whether interactive PPG signal plots are needed and for which participants
# if empty, all participants in `ppg_dict` will be used
interactive_ppg_plot = True
participants_plots = []  # set specific ID(s) here if needed, e.g., ['sub-802']. Leave empty for all.

# For S5: set whether (optional) manual correction of PPG peaks using Systole's Editor is needed
manual_correct = True

# For S6: set whether data export of TSV file with interpolated HR values (in bpm) is needed
hr_interpol = True

## **1. Data import and conversion**

This section imports the physiological data and metadata from the `_physio.tsv.gz` and sidecar `_physio.json` files and extracts the raw PPG data as a numpy array (`ppg_raw`). This array is added to a dictionary (`ppg_dict`) where the data is organized by participant. In detail:

- **Define participants and BIDS file paths**: first, the user specifies the participant ID(s) in the `participant_ids` list. If `participant_ids` is empty, the script automatically includes all participants in the main directory of BIDS-compliant raw data storage (`wd`). The user must specify the root directory of BIDS-compliant raw data storage (`wd`), as well as the mandatory (i.e., task label, datatype) and optional (i.e. session label) BIDS entities (in the format `task-<label>`, `<datatype>` and `ses-<label>`, respectively). These will be used to create a base filename according to BIDS conventions (e.g., `sub-<ID>{_ses-<label>}_task-<label>`) and a base BIDS directory including subject, session (optional) and datatype (e.g., `'sub-<ID>/[ses-<label>/]<datatype>/'`). 
- **Check for `_physio.tsv.gz` and `_physio.json` file existence**: ensures the raw PPG signal and metadata exist for each participant. If either file is missing, the participant is skipped with a warning message.
- **Extract and parse metadata**: read the JSON file to extract key information, including sampling frequency (saved as `sfreq`) and column names used to recognize the PPG data column when reading the TSV.GZ file (expects a column named `ppg`). Additionally reads and decompresses the TSV.GZ file into a pandas DataFrame (`physio_df`) using the extracted column names.
- **Organize PPG data into a dictionary**: the following data from each participant is stored in a dictionary (`ppg_dict`) with the participant IDs (`sub-<label>`) as keys: 
    - `bids_base_filename`: the base PPG filename in BIDS format. 
    - `bids_base_directory`: the base BIDS-compliant folder structure with or without session entity, i.e., `'sub-<ID>/{ses-<label>}/<datatype>/'`. 
    - `ppg_raw`: the raw PPG signal stored as an array.
    - `sfreq`: the sampling frequency of the PPG recording.
- **Define the derivatives path for PPG preprocessed data storage**: defines and creates the directory for storing the PPG preprocessed data, i.e., `derivatives/ppg-preproc/sub-<ID>/...`. 

In [ ]:
################## 1. Data import and conversion ##################

############## Define path for PPG data ##############

# Set participant IDs
# If set to an empty list '[]', it will process all participants in the directory
participant_ids = ['201', '202']  # Adjust as needed: each item should correspond to <ID> of 'sub-<ID>' in BIDS format; otherwise, leave empty for all

# Specify the main directory of data storage (containing BIDS-compatible raw data)
wd = r'C:\Users\gerosa\Desktop\BBSIG_datasets\PPG'  # change with the directory of data storage

# Mandatory: BIDS entities (task, datatype)
task_name = 'BBSIG'       # <label> of 'task-<label>' used for file naming in BIDS format
datatype_name = 'beh'     # datatype used for corresponding directory in BIDS format (e.g., 'beh', 'eeg', 'func')
physio_name = 'physio'    # physio data specification in BIDS format

# Optional: BIDS entities (session)
session_idx = '01'     # <label> of 'ses-<label>' in BIDS format, if available; otherwise, set to None


############## Load raw PPG data ##############

# Initialize dict for storing 
ppg_dict = {}

# Get all participant IDs if none are specified
if not participant_ids:
    # Assuming participant folders are named 'sub-<ID>'
    rawdata_dir = os.path.join(wd)
    participant_ids = [d.replace('sub-', '') for d in os.listdir(rawdata_dir) if d.startswith('sub-')]

# Iterate through each participant
for subj in participant_ids:

    subj_id = 'sub-' + str(subj)  # participant ID (in BIDS format)

    ############## Create base BIDS-compatible filename and directory ##############

    # If you have additional BIDS entities (e.g., 'run' or 'recording') you can change the base BIDS file name below
    # e.g., f'{subj_id}_ses-{session_idx}_task-{task_name}_run-{run_idx}_recording-{rec_name}' 
    if session_idx is not None: 
        bids_base_fname = f'{subj_id}_ses-{session_idx}_task-{task_name}' 
        bids_base_dir = os.path.join(subj_id, f'ses-{session_idx}', datatype_name) # base BIDS folders incl. session 'sub-<ID>/ses-<label>/<datatype>/'
    else:
        bids_base_fname = f'{subj_id}_task-{task_name}'
        bids_base_dir = os.path.join(subj_id, datatype_name) # base BIDS folders without session 'sub-<ID>/<datatype>/'
    

    # Specify the directory and name of the BIDS-compatible physio data TSV.GZ file
    tsv_fname = f'{bids_base_fname}_{physio_name}.tsv.gz'
    tsv_dir = os.path.join(wd, bids_base_dir, tsv_fname)

    # Specify the name and directory of the BIDS-compatible physio metadata JSON file
    json_fname = f'{bids_base_fname}_{physio_name}.json'
    json_dir = os.path.join(wd, bids_base_dir, json_fname)

    # Check if the files exist for the participant
    if not os.path.exists(tsv_dir) or not os.path.exists(json_dir):
        print(f"Data files for {subj_id} not found. Skipping.")
        continue
    
    # Extract physio metadata from JSON file
    with open(json_dir, 'r') as fjson:
        physio_metadata = json.load(fjson)  # load metadata from physio.json
    physio_col = physio_metadata['Columns']  # column names from tsv.gz file 

    # Open physio data file and save as DataFrame
    physio_df = pd.read_csv(tsv_dir, compression='gzip', header=None, index_col=False,
                            sep='\t', names=physio_col)     # decompress and read TSV file
    sfreq = physio_metadata['SamplingFrequency']  # save sampling rate

    # Extract PPG data from physio_df and save it as ppg_df and ppg_arr
    physio_df.columns = [col.lower() for col in physio_df.columns]
    if 'ppg' in physio_df.columns:
        ppg_df = physio_df['ppg']   # save PPG df (one data sample per row)
        ppg_arr = ppg_df.values     # save array with PPG data samples

    # Store the data in a dictionary with the participant ID as key
    ppg_dict[subj_id] = {
        'bids_base_filename' : bids_base_fname, # tmp for BIDS base filename
        'bids_base_directory': bids_base_dir,   # tmp for BIDS base directory
        'ppg_raw' : ppg_arr,
        'sfreq' : sfreq
    }

    ############## Define path for PPG preprocessed data ##############

    # Check if the BIDS directory 'derivatives/ppg-preproc/sub-<ID>/...' exists, if not create it
    ppg_preproc_folder = 'ppg-preproc'
    ppg_preproc_dir = os.path.join(wd, 'derivatives', ppg_preproc_folder, bids_base_dir)
    if not os.path.exists(ppg_preproc_dir):
        os.makedirs(ppg_preproc_dir)

## **2. (Optional) PPG normalization, filtering & clipping artifacts correction**

This section includes a series of optional preprocessing steps for PPG signal correction and cleaning. In detail: 

2a. **PPG normalization**: if the variable `ppg_normalize` is set to `True` in the optional pipeline steps (see settings above - defaults to `True`), this part of the code will:
- Define a custom function to **normalize the PPG signal between -1 and 1**. 
- Perform the normalization for all participants in `ppg_dict` and store the normalized PPG signal under the `ppg_norm` key.  

2b. **PPG filtering**: If the variable `ppg_filter` is set to `True` in the optional pipeline steps (see settings above - defaults to `True`), this part of the code will:
- Apply filtering to the PPG signal of each participant using NeuroKit2's `nk.ppg_clean()` function. The filtering method defaults to `elgendi`, as recommended for general-purpose preprocessing.
- Store the filtered signal for each participant in `ppg_dict()` under the `ppg_filt` key.

2c. **PPG clipping artifacts detection and interpolation**: If the variable `clip_artifacts_correct` is set to `True` in the optional pipeline steps (see settings above), this part of the code will:
- Detect clipping artifacts at a minimum and/or maximum threshold using Systole's `find_clipping()` function.
- Interpolate over the detected artifacts with `interpolate_clipping`, using cubic interpolation by default. 
- Save the corrected PPG signal under the `ppg_clipping_clean` key in `ppg_dict`. If clipping artifacts are detected and interpolated for a given participant, this information will be stored under the key `ppg_clipping_interpolation` as `True`. 

If none of these preprocessing steps are enabled, all the subsequent sections will rely on the raw PPG signal (`ppg_raw`) instead. Otherwise, they will be carried out in order of preference using the `ppg_clipping_clean`, `ppg_filt` and `ppg_norm` signal. 

In [ ]:
################## 2a. (Optional) PPG normalization ##################

# Define custom function to normalize the PPG signal between -1 and 1
def normalize_ppg_signal(ppg_signal):
    """
    Function to normalize the PPG signal between -1 and 1.
    Args:
        ppg_signal (array): The raw PPG signal to normalize.
    Returns:
        ppg_norm (array): The normalized PPG signal. 
    """
    
    # Get min and max values from PPG signal
    ppg_min = np.min(ppg_signal)
    ppg_max = np.max(ppg_signal)
    
    # Perform the normalization between -1 and 1
    ppg_norm = 2 * ((ppg_signal - ppg_min) / (ppg_max - ppg_min)) - 1
    
    return ppg_norm

In [ ]:
# Proceed only if (optional) pre-processing step of PPG normalization is required
if ppg_normalize: 

    # Iterate through the extracted data dictionary and normalize the PPG signal
    for participant_id, data in ppg_dict.items():
        ppg_raw = data['ppg_raw']
        
        if ppg_raw is not None:

            # Normalize the PPG signal
            ppg_norm = normalize_ppg_signal(ppg_raw)
            
            # Update the dictionary with the normalized signal
            ppg_dict[participant_id]['ppg_norm'] = ppg_norm

In [ ]:
################## 2b. (Optional) PPG filtering ##################

# Proceed only if (optional) pre-processing step of PPG filtering is required
if ppg_filter:
    
    for participant_id, data in ppg_dict.items():
            
        # Get normalized PPG signal if available, otherwise use raw signal
        ppg_signal = data.get('ppg_norm' if ppg_normalize else 'ppg_raw')
        sfreq = data.get('sfreq')

        # Filter the PPG signal using NeuroKit2's ppg_clean() function with 'elgendi' default method
        signal_clean = nk.ppg_clean(ppg_signal=ppg_signal, sampling_rate=sfreq, method='elgendi')
            
        # Save the filtered PPG signal to the dictionary
        ppg_dict[participant_id]['ppg_filt'] = signal_clean

In [ ]:
################## 2c. (Optional) PPG clipping artifacts correction ##################

# Proceed only if (optional) pre-processing step of correcting PPG clipping artifacts is required
if clip_artifacts_correct:
    
    for participant_id, data in ppg_dict.items():

        sfreq = data.get('sfreq')

        # Choose the PPG signal based on filtering
        ppg_signal = data.get(
            'ppg_filt' if ppg_filter else
            'ppg_norm' if ppg_normalize else 'ppg_raw')
        
        # Find max and/or min threshold of clipping artifacts, if present, otherwise return None
        min_threshold, max_threshold = find_clipping(signal=ppg_signal)

        # Clean PPG signal from clipping artifacts filling missing segments via interpolation (default = 'cubic')
        ppg_clip_clean = interpolate_clipping(signal=ppg_signal, 
                                              min_threshold=min_threshold, 
                                              max_threshold=max_threshold)
        
        # Save the PPG cleaned from clipping artifacts to the dictionary
        ppg_dict[participant_id]['ppg_clipping_clean'] = ppg_clip_clean

        if (min_threshold is not None) or (max_threshold is not None): 
            ppg_dict[participant_id]['ppg_clipping_corrected'] = 'True'

            print(f"Clipping artifacts corrected for {participant_id}")
        else:
            ppg_dict[participant_id]['ppg_clipping_corrected'] = 'False'

## **3. PPG peak detection**

This section performs **systolic peak detection** on the provided PPG signal and **optionally applies automated correction**, based on the chosen method. In detail, this section: 

- Defines a custom function `detect_ppg_peaks()` for detecting systolic peaks using NeuroKit2's `ppg_peaks()` with the default method `'elgendi'` (for best results, this method expects the filtered PPG signal, so make sure that `ppg_filter` is set to `True` to execute Sect. 2b). Alternatively, PPG peaks detection can be implemented using the method `'bishop'` (suitable only for short time-windows and low sampling frequency e.g., 5 seconds and 100 Hz). 
- Store the indices of the uncorrected systolic peaks in the `'ppg_peaks_info'` dictionary as `'PPG_Peaks_Uncorrected'`. 
- Optionally, implement two complementary methods for artifact correction: 
    - If `correct_artifacts_nk` is set to `True`, it enables artifact correction by Systole or by the automated artifact correction built in to NeuroKit2’s `ppg_peaks()` function.
    - If `correct_artifacts_sys` is set to `True`, it enables artifact correction with Systole's `correct_peaks()`, which is based on the detection of RR-interval abnormalities. The detection-correction process will be repeated as many times as specified by the `iterations` variable (see settings above; defaults to 1). The indices of the uncorrected artifact types, as well as the number of extra and missed peaks corrected with this method, will be saved in the 'info' dictionary. 

In [ ]:
################## 3. PPG peak detection ##################

# Define a custom function for PPG peak detection:
# using NeuroKit2 ppg_peaks(method='elgendi') as default, with automated artifact correction, 
# plus (optional) additional step of peak correction using Systole's correct_peaks()

def detect_ppg_peaks(signal, sfreq, method='elgendi', correct_artifacts_nk=correct_artifacts_nk, 
                      correct_artifacts_sys=correct_artifacts_sys, n_iterations=iterations):
    """
    Detect R-peaks in PPG signal using the specified method with optional artifact correction.

    Parameters:
    -----------
    signal : array-like
        The PPG signal (e.g., 'ppg_clean' if filtered, or 'ppg_raw' if raw).
    sfreq : float
        The sampling frequency of the PPG signal.
    method : {'elgendi', 'bishop'}, optional
        The method used for R-peak detection. Default is 'elgendi'.
    correct_artifacts_nk : bool, optional
        Whether to apply automated artifacts correction in the PPG peak detection, 
        using NeuroKit2 method by Lipponen & Tarvainen (2019). Default is True.
    correct_artifacts_sys : bool, optional
        Whether to apply PPG peak correction using Systole's correct_peaks(). Default is True.
    n_iterations : int, optional
        If correct_artifacts_sys is True, specify how many iterations to perform. Default is 1.

    Returns:
    --------
    dict containing:
        - 'ppg_peaks_idx' (array): Indices of the detected R-peaks.
        - 'ppg_peaks_bool' (array): Boolean array indicating the presence of R-peaks.
        - 'ppg_peaks_info' (dict): Additional info about the detection and correction methods, artifacts, corrected artifacts, and uncorrected peaks.
    """
    
    # Initialize info dict with detection method
    info = {
        'method_peaks': method,
        'peaks_correction_neurokit': str(correct_artifacts_nk),
        'peaks_correction_systole': str(correct_artifacts_sys)
    }

    # Detect PPG peaks without any automated peaks correction
    uncorr_peaks_df, uncorr_peaks_info = nk.ppg_peaks(signal, sampling_rate=sfreq, 
                                                      method=method, correct_artifacts=False)
    info['PPG_Peaks_Uncorrected'] = uncorr_peaks_info['PPG_Peaks']
    uncorr_peaks_bool = np.array(uncorr_peaks_df['PPG_Peaks'])

    # Temporary saving boolean and idx of uncorrected peaks
    peaks_bool_tmp = uncorr_peaks_bool
    peaks_idx_tmp = uncorr_peaks_info['PPG_Peaks']

    # Detect artifacts in RR series (ectopic, long, short, missed, extra)
    artifacts = rr_artefacts(peaks_bool_tmp, input_type='peaks')

    # Convert each artifact type from bool to idx before applying correction
    artifact_types = ['ectopic', 'long', 'short', 'extra', 'missed']
    for artifact in artifact_types:
        idx = input_conversion(artifacts[artifact], input_type='peaks', output_type='peaks_idx', sfreq=sfreq)
        info[f'{artifact}_idx_uncorr'] = idx

    # Optional: apply PPG peak detection using NeuroKit2's automated artifacts correction
    # if correct_artifacts_nk is set to True (default)
    if correct_artifacts_nk:
        peaks_nk_df, peaks_nk_info = nk.ppg_peaks(signal, sampling_rate=sfreq, method=method, 
                                                  correct_artifacts=correct_artifacts_nk)   
        peaks_nk_bool = np.array(peaks_nk_df['PPG_Peaks'])
        peaks_nk_idx = peaks_nk_info['PPG_Peaks']

        # Temporary saving boolean and idx of NeuroKit corrected peaks
        peaks_bool_tmp = peaks_nk_bool
        peaks_idx_tmp = peaks_nk_idx

    # Optional: apply artifacts correction using Systole correct_peaks() function, 
    # if correct_artifacts_sys is set to True (default)
    if correct_artifacts_sys:
        
        # Check if automated artifacts correction using NeuroKit2 was previously performed
        peaks_bool = peaks_nk_bool if correct_artifacts_nk else uncorr_peaks_bool
        
        # Correct artifacts using Systole’s correct_peaks() function
        info_correction = correct_peaks(peaks_bool, n_iterations=n_iterations) # add verbose=False to stop from printing steps
        peaks_sys_bool = info_correction['clean_peaks']
        peaks_sys_idx = input_conversion(peaks_sys_bool, input_type='peaks', output_type='peaks_idx', sfreq=sfreq)
        
        # Add correction details to the info dictionary
        info['info_correction_systole'] = {
            'extra': info_correction['extra'],
            'missed': info_correction['missed']
        }
        
        # Temporary saving boolean and idx of Systole corrected peaks
        peaks_bool_tmp = peaks_sys_bool
        peaks_idx_tmp = peaks_sys_idx
    
    return {'ppg_peaks_idx': peaks_idx_tmp, 
            'ppg_peaks_bool': peaks_bool_tmp, 
            'ppg_peaks_info': info}

In [ ]:
# Perform the PPG systolic peak detection with the chosen (optional) automated correction methods

# Check that PPG signal has been filtered before using the 'elgendi' method for peak detection
if not ppg_filter:
    print("Attention! If using the default method 'elgendi' for peak detection, make sure to execute section 2b first, by setting 'ppg_filter' to True.")

# Iterate over participants and process the PPG data
for participant_id, data in ppg_dict.items():
    
    sfreq = data.get('sfreq')

    # Choose the PPG signal based on the last pre-processing step performed 
    ppg_key = ('ppg_clipping_clean' if clip_artifacts_correct else 
               'ppg_filt' if ppg_filter else 
               'ppg_norm' if ppg_normalize else 'ppg_raw')
    ppg_signal = data.get(ppg_key)

    # Print which PPG signal was selected for systolic peak detection
    print(f'Participant {participant_id} - Selected PPG signal for peaks detection: {ppg_key}')
    
    # Call the function for PPG peak detection and (optionally) artifact correction
    ppg_peaks_result = detect_ppg_peaks(signal=ppg_signal, sfreq=sfreq, method='elgendi', 
                                         correct_artifacts_nk=correct_artifacts_nk, 
                                         correct_artifacts_sys=correct_artifacts_sys, n_iterations=iterations)

    # Extract the values
    peaks_idx = ppg_peaks_result['ppg_peaks_idx']
    peaks_bool = ppg_peaks_result['ppg_peaks_bool']
    info = ppg_peaks_result['ppg_peaks_info']
        
    # Assign the peaks_bool as a boolean column in ppg_df
    ppg_dict[participant_id]['ppg_peaks_bool'] = peaks_bool.astype(int)  # Convert boolean array to 1s and 0s for binary representation

    # Store peaks_idx on the Dictionary (different length, cannot be saved on df)
    ppg_dict[participant_id]['ppg_peaks_idx']= peaks_idx
    
    # Store info as a string representation in the Dictionary
    ppg_dict[participant_id]['ppg_peaks_info'] = info

## **4. (Optional) interactive visualization**

If `interactive_ppg_plot` is set to `True`, this section provides two complementary types of interactive visualization of the PPG signal, using Systole's `plot_raw()` and `plot_subspaces()` functions. If `plot_within_notebook` is set to `True`, the interactive plots for all participants will be rendered within the notebook using Bokeh as the backend, otherwise each plot will be opened as separate HTML file in the browser (note that this is the recommended option when processing many participants at a time). In detail: 
- **4a. Interactive plot of PPG signal and systolic peaks**: display an interactive plots of PPG signal over time with systolic peaks and instantaneous heart rate using Systole's `plot_raw()`. 
- **4b. Interactive plot of subspaces**: display an interactive visualization of PPG subspaces plots, including short/long intervals and ectopic beats using Systole's `plot_subspaces()`, based on the artifact detection method described in Lipponen & Tarvainen (2019).

In [ ]:
############## 4a. Interactive plot of PPG signal and systolic peaks ##############

# Select whether to visualize interactive plots within notebook using Bokeh
# Otherwise, each plot will be opened in browser as .html file 
plot_within_notebook = True

# Proceed only if (optional) interactive plotting of PPG signal and peaks is required
if interactive_ppg_plot:

    # If selected, activate Bokeh to display plots in the notebook
    if plot_within_notebook:
        output_notebook()

    # Determine which participants to include based on `participant_plots` subset,
    # if empty, take all participant IDs from ppg_dict 
    participants_to_plot = participants_plots if participants_plots else list(ppg_dict.keys())

    # Iterate over the selected participants and process the PPG data
    for subj_id in participants_to_plot:
        data = ppg_dict.get(subj_id)
        if not data:
            print(f"No data found for participant {subj_id}. Skipping.")
            continue

        sfreq = data.get('sfreq') # get sampling frequency

        # Choose the PPG signal based on pre-processing steps
        # Prefer in order clipping artifacts correction, filtering, normalization and raw
        ppg_signal = data.get(
            'ppg_clipping_clean' if clip_artifacts_correct else
            'ppg_filt' if ppg_filter else
            'ppg_norm' if ppg_normalize else 'ppg_raw')
        
        peaks = data.get('ppg_peaks_bool') # get systolic peaks as bool 
        
        # Show the plot with continuous PPG data, systolic peaks, HR and other options
        plot = plot_raw(signal=ppg_signal,peaks=peaks.astype(bool), modality="ppg", sfreq=sfreq,
                        backend="bokeh", show_artefacts=True, slider=True, show_heart_rate=True)

        show(plot)

In [ ]:
############## 4b. Plot subspaces with short/long intervals and ectopic beat ##############

# Proceed only if (optional) interactive plotting of PPG signal and peaks is required
if interactive_ppg_plot:

    # If selected, activate Bokeh to display plots in the notebook
    if plot_within_notebook:
        output_notebook()
    
    # Determine which participants to include based on `participant_plots` subset,
    # if empty, take all participant IDs from ppg_dict 
    participants_to_plot = participants_plots if participants_plots else list(ppg_dict.keys())

    # Iterate over the selected participants and process the PPG data
    for subj_id in participants_to_plot:
        data = ppg_dict.get(subj_id)
        if not data:
            print(f"No data found for participant {subj_id}. Skipping.")
            continue

        sfreq = data.get('sfreq') # get sampling frequency

        # Choose the PPG signal based on pre-processing steps
        # Prefer in order clipping artifacts correction, filtering, normalization and raw
        ppg_signal = data.get(
            'ppg_clipping_clean' if clip_artifacts_correct else
            'ppg_filt' if ppg_filter else
            'ppg_norm' if ppg_normalize else 'ppg_raw')
        
        peaks = data.get('ppg_peaks_bool') # get systolic peaks as bool 
        
        # Show the plot with PPG data, R-peaks, and other options
        show(
            plot_subspaces(
                rr=peaks.astype(bool), backend='bokeh', input_type='peaks'))

## **5. Manual PPG peak correction**

If enabled via `manual_correct`, this section triggers the **interactive manual correction of PPG systolic peak locations and identification of noisy segments in the PPG signal** using Systole's `Editor`. Both the raw PPG signal and the instantaneous heart rate are plotted to check for artifacts (e.g., long/short beats, ectopic beats). This interactive plot features a "Correction" mode for deleting peaks or adding them at the local maxima within selected segments, and a "Rejection" mode for marking selected segments as 'bad'. 

- In Correction mode: click and drag the *left mouse button* to select a segment where all the peaks should be removed; click and drag the *right mouse button* to select a segment where a peak will be added at the local maximum.
- In Rejection mode: click and drag the *right mouse button* to select a segment that should be marked as a bad segment. This will be saved as a pair of indices indicating the onset and offset of the bad segment. Important: you cannot undo bad segments!

It is recommended to **perform manual correction one participant at a time**, by specifying the desired participant ID in the list `participants_manual = []`. Once you are done with manual correction for one participant and have saved the corresponding JSON file by running Sect. 5b, you can change the participant ID and re-run the entire manual correction section again. Note that, after manually correcting a few participants, the interactive plot might become laggy or freeze, so you might want to run the entire preprocessing pipeline only on a handful of participants at a time. 

In [ ]:
############## 5. PPG peak manual correction ##############

# For smoother processing, we recommend to perform manual correction of PPG peaks using 
# Systole's Editor one participant at a time
if manual_correct:
    participants_manual = ['sub-203'] # change with desired participant ID

    print(f'Manual correction of PPG peaks will be presented for participant: {participants_manual}')

In [ ]:
############# 5a. Interactive plot for manual peak correction ##############

if manual_correct:
    
    # Determine participants to correct
    participants_to_correct = participants_manual if participants_manual else print('Please select at least one participant to manually correct above')

    # Iterate over selected participants for manual correction
    for subj_id in participants_to_correct:
        data = ppg_dict.get(subj_id)
        if not data:
            print(f'No data found for participant {subj_id}. Skipping.')
            continue

        sfreq = data.get('sfreq') # get sampling frequency

        # Choose the PPG signal based on pre-processing steps
        # Prefer in order clipping artifacts correction, filtering, normalization and raw
        ppg_signal = data.get(
            'ppg_clipping_clean' if clip_artifacts_correct else 
            'ppg_filt' if ppg_filter else 
            'ppg_norm' if ppg_normalize else 'ppg_raw')

        # Define paths for saving corrected JSON files per participant
        if session_idx is not None: 
            participant_preproc_dir = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, f'ses-{session_idx}', datatype_name)
        else:
            participant_preproc_dir = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, datatype_name)
            
        # Define the corrected JSON file path
        bids_base_fname = data.get('bids_base_filename')
        manualcorr_fname = f'{bids_base_fname}_manual-correction.json'
        manualcorr_fpath = os.path.join(participant_preproc_dir, manualcorr_fname)

        # Display interactive plot for manual correction
        # Display interactive plot
        %matplotlib ipympl

        editor = Editor(signal=ppg_signal, 
                        corrected_json=manualcorr_fpath,
                        sfreq=sfreq, 
                        corrected_peaks=data.get('ppg_peaks_bool').astype(bool),
                        signal_type='PPG', figsize=(10, 6))

        display(editor.commands_box)

In [ ]:
############# 5b. Interactive plot for manual peak correction: data saving ##############

# After completing manual correction, save the corrected file
if manual_correct:
    editor.save()

## **6. Data output**

This section exports the PPG preprocessing output files in BIDS-compliant format for each subject in `derivatives/ppg-preproc/sub-<label>/<datatype>/`.

- **6a. (Optional) Export TSV.GZ file with raw and clean PPG data**: a custom function, `save_ppg_cleaned()`, saves the BIDS-compliant `_ppg-cleaned.tsv.gz` file with two columns: `ppg_raw` (the original PPG data) and `ppg_cleaned` (the cleaned PPG signal, created by the optional steps in Sect. 2). This ensures easy access for later stages of analysis and enhances reproducibility. 

- **6b. Export JSON file with main PPG-preprocessing features**: a custom function `save_ppg_preproc()` saves the `_ppg-preproc.json` file containing the following information:
    - **`ppg_peaks`** contains the following data: `PPG_Peaks_Uncorr` for uncorrected PPG peak indices; `PPG_Peaks_AutoCorr` for auto-corrected PPG peak indices, if either NeuroKit2's or Systole's artifact correction were used; `PPG_Peaks_ManualCorr` for manually corrected PPG peak indices using Systole's `Editor`. 
    - **`rr_s`** contains the RR interval time series (in seconds) created using Systole's `input_conversion(output_type='rr_s')`, based on the uncorrected, auto-corrected and/or manually corrected PPG systolic peak indices (if present). 
    - **`bad_segments`** contains index pairs indicating the onsets and offsets of PPG signal segments marked as "bad" using Systole's `Editor`.
    - **`info`** contains metadata about the PPG peak detection procedure, including the methods chosen for peak detection and artifact correction, the indices of uncorrected artifacts and the number of extra/missed peaks automatedly corrected. 

- **6c. (Optional) Export TSV.GZ file with interpolated HR (in bpm)**: if enabled under the `hr_interpol` optional setting, a custom function, `save_hr_interpol()`, saves the **interpolated heart rate (HR) values in BPM from the RR interval time series** with the selected correction type (i.e., `manualcorr` > `autocorr` > `uncorr`) to a new file ending in `_hr-bpm-{correction_type}.tsv.gz`.  Please note that interpolated HR values before the first RR interval and after the last RR interval will be filled with NaN values.

In [ ]:
############## 6. Data output ##############

############# 6a. (Optional) Create and export TSV.GZ file with raw and clean PPG ##############
def save_ppg_cleaned(ppg_raw, ppg_clean, subj_id, bids_base_fname):
    """ Save raw and cleaned PPG data to a TSV.GZ file.

    Parameters:
    - ppg_raw (ndarray): Array containing raw PPG data (i.e., same as BIDS-compatible _physio.tsv.gz)
    - ppg_clean (ndarray): Array containing cleaned PPG data. Based on the optional pre-preprocessing steps, 
        it could correspond to 'ppg_clipping_clean', 'ppg_filt', or 'ppg_norm'. 
    - subj_id (str): ID of the participant.
    - bids_base_fname (str): The base PPG filename in BIDS format. 

    Raises:
    - ValueError: If ppg_clean is None.
    """
    if ppg_clean is None:
        raise ValueError("ppg_clean cannot be None. Provide a valid cleaned PPG array.")
    
    # Merge ppg_raw and ppg_clean into one dataframe
    ppg_all = pd.DataFrame({'ppg_raw': ppg_raw, 'ppg_cleaned': ppg_clean})

    # Save as TSV.GZ in 'derivatives/ppg-preproc/sub-XX/<datatype>/' folder
    ppg_all_fname = f'{bids_base_fname}_ppg-cleaned.tsv.gz'

    if session_idx is not None: 
        ppg_all_fpath = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, 
                                     f'ses-{session_idx}', datatype_name, ppg_all_fname)
    else:
        ppg_all_fpath = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, 
                                       datatype_name, ppg_all_fname)
    os.makedirs(os.path.dirname(ppg_all_fpath), exist_ok=True)

    ppg_all.to_csv(ppg_all_fpath, index=False, compression='gzip', 
                   sep='\t', na_rep="n/a")
    
    print(f"PPG cleaned data for {subj_id} saved at {ppg_all_fpath}")

In [ ]:
############# 6b. Export main PPG preprocessing info to JSON file  ##############

# Helper function to convert values to int or None
def convert_to_int_or_none(value):
    if isinstance(value, (float, int, np.float64, np.int64)):
        return int(value) if not np.isnan(value) else None
    return None

# Main function for PPG preprocessing data export
def save_ppg_preproc(ppg_dict, subj_id, bids_base_fname, manual_correct=False, hr_interpol=False):
    """
    Save PPG preprocessing information to a JSON file.

    Parameters:
    - peaks_dict (dict): dictionary containing PPG peak detection info for one participant. 
    - subj_id (str): the participant ID.
    - bids_base_fname (str): The base PPG filename in BIDS format. 
    - manual_correct (bool): whether manual correction was performed with Systole Editor.
    - hr_interpol (bool): whether to compute RR intervals for interpolation.

    Returns:
    - dict: containing all relevant PPG preprocessing information.
    """
    # Initialize dictionary for saving PPG preprocessing info
    ppg_preproc_dict = {
        'ppg_peaks': {}, 
        'rr_s': {},  
        'bad_segments': {}, 
        'info': {}
    }

    sfreq = ppg_dict.get('sfreq') # get sampling frequency
    
    # Extract uncorrected PPG peaks 
    peaks_uncorr = ppg_dict['ppg_peaks_info'].get('PPG_Peaks_Uncorrected').tolist()
    ppg_preproc_dict['ppg_peaks']['PPG_Peaks_Uncorr'] = peaks_uncorr

    # Extract PPG peaks automatedly corrected with NeuroKit2 and/or Systole 
    if correct_artifacts_nk or correct_artifacts_sys:
        peaks_auto_corr = ppg_dict['ppg_peaks_idx'].tolist()
        ppg_preproc_dict['ppg_peaks']['PPG_Peaks_AutoCorr'] = peaks_auto_corr
    
    # Store RR intervals derived from uncorrected and auto corrected peaks
    rr_s_uncorr = input_conversion(peaks_uncorr, input_type='peaks_idx', output_type='rr_s', sfreq=sfreq)
    ppg_preproc_dict['rr_s']['RR_s_Uncorr'] = rr_s_uncorr.tolist()

    rr_s_auto_corr = input_conversion(peaks_auto_corr, input_type='peaks_idx', output_type='rr_s', sfreq=sfreq)
    ppg_preproc_dict['rr_s']['RR_s_AutoCorr'] = rr_s_auto_corr.tolist()

    # Apply manual correction if specified
    if manual_correct:

        # Define path for manually corrected JSON files per participant
        manualcorr_fname = f'{bids_base_fname}_manual-correction.json'
        if session_idx is not None: 
            participant_manualcorr_dir = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, 
                                                      f'ses-{session_idx}', datatype_name, manualcorr_fname)
        else:
            participant_manualcorr_dir = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, 
                                                      datatype_name, manualcorr_fname)
        if os.path.exists(participant_manualcorr_dir):
            with open(participant_manualcorr_dir, 'r') as fsyst:
                mancorr_data = json.load(fsyst)
            
            # Extract manually corrected peaks and bad segments
            peaks_manual_corr = mancorr_data.get('ppg', {}).get('corrected_peaks', [])
            bad_segments_idx = mancorr_data.get('ppg', {}).get('bad_segments', [])
            ppg_preproc_dict['ppg_peaks']['PPG_Peaks_ManualCorr'] = peaks_manual_corr
            ppg_preproc_dict['bad_segments'] = bad_segments_idx
            
            # Store RR intervals derived from manually corrected peaks
            rr_s_manual_corr = input_conversion(peaks_manual_corr, input_type='peaks_idx', output_type='rr_s', sfreq=sfreq)
            ppg_preproc_dict['rr_s']['RR_s_ManualCorr'] = rr_s_manual_corr.tolist()

        else:
            print(f'Manual correction file not found for {subj_id}')    

    # Extract and filter useful metadata from info
    info_dict = ppg_dict['ppg_peaks_info']
    filtered_info = {
        k: (v.tolist() if isinstance(v, np.ndarray) else v)
        for k, v in info_dict.items()
        if k in [
            'method_peaks', 'peaks_correction_neurokit', 
            'peaks_correction_systole', 'extra_idx_uncorr', 'missed_idx_uncorr',
            'ectopic_idx_uncorr', 'long_idx_uncorr', 'short_idx_uncorr', 
            'info_correction_systole'
        ]
    }
    ppg_preproc_dict['info'] = filtered_info

    # Define file path and save JSON file with PPG preprocessing info
    ppg_preproc_json_fname = f'{bids_base_fname}_ppg-preproc.json'
    if session_idx is not None: 
        ppg_preproc_json_fpath = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, 
                                              f'ses-{session_idx}', datatype_name, ppg_preproc_json_fname)
    else:
        ppg_preproc_json_fpath = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, 
                                              datatype_name, ppg_preproc_json_fname)
    os.makedirs(os.path.dirname(ppg_preproc_json_fpath), exist_ok=True)
    with open(ppg_preproc_json_fpath, 'w') as ppg_preproc_json:
        json.dump(ppg_preproc_dict, ppg_preproc_json, allow_nan=True, indent=4)

    print(f'PPG preprocessing results for {subj_id} have been saved to {ppg_preproc_json_fpath}')

    return ppg_preproc_dict

In [ ]:
############# 6c. (Optional) Export TSV.GZ file with interpolated HR (in bpm) ##############

def save_hr_interpol(ppg_preproc_dict, sfreq, subj_id, bids_base_fname, interpol_type='cubic'):
    """
    Save interpolated heart rate (HR) values to a TSV.GZ file.

    Parameters:
    - ppg_preproc_dict (dict): dictionary containing info about PPG preprocessing, generated by save_ppg_preproc().
    - sfreq (float): sampling frequency of the RR intervals.
    - subj_id (str): ID of the participant.
    - bids_base_fname (str): The base PPG filename in BIDS format. 
    - interpol_type (str, optional): type of interpolation to apply. Default is 'cubic'.

    Returns:
    - pd.DataFrame: DataFrame containing the interpolated HR values.
    """

    # Determine the RR type to use based on availability
    rr_key_priority = ['RR_s_ManualCorr', 'RR_s_AutoCorr', 'RR_s_Uncorr']
    rr_key = next((key for key in rr_key_priority if key in ppg_preproc_dict['rr_s']), None)

    rr_key_method = rr_key.split('_')[-1].lower() if rr_key else None
    
    if not rr_key:
        print(f'No valid RR intervals found for participant {subj_id}. Skipping HR interpolation.')
        return None

    # Use the selected RR intervals
    rr_tmp = ppg_preproc_dict['rr_s'][rr_key]

    # Check if interpol_type is valid
    if interpol_type not in ['cubic', 'linear', 'previous', 'next']:
        raise ValueError("Invalid interpolation type. Please select either 'cubic' (default), 'linear', 'previous', or 'next'.")

    # Interpolate heart rate values using Systole heart_rate()
        # Hot fix @April 3rd, 2025: removed sfreq=sfreq from arguments due to bug with Systole mis-calculating HR (until it will be fixed)
    hr_bpm, hr_time = heart_rate(x=rr_tmp, unit='bpm', kind=interpol_type, input_type='rr_s')

    # Define directory for TSV file with interpolated HR values
    hr_tsv_fname = f'{bids_base_fname}_hr-bpm-{rr_key_method}.tsv.gz'
    if session_idx is not None: 
        hr_tsv_fpath = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, 
                                    f'ses-{session_idx}', datatype_name, hr_tsv_fname)
    else:
        hr_tsv_fpath = os.path.join(wd, 'derivatives', ppg_preproc_folder, subj_id, 
                                    datatype_name, hr_tsv_fname)
    os.makedirs(os.path.dirname(hr_tsv_fpath), exist_ok=True)

    # Save TSV.GZ file with interpolated HR values (in bpm)
    hr_df = pd.DataFrame(hr_bpm, columns=[f"hr_bpm_{rr_key_method}"])
    hr_df.to_csv(hr_tsv_fpath, index=False, compression='gzip', sep='\t', na_rep="n/a")

    print(f"HR interpolation results for {subj_id} have been saved to {hr_tsv_fpath}")

    return hr_df

In [ ]:
# Loop through each participant in ppg_dict and apply the preprocessing and saving functions
for subj_id, data in ppg_dict.items():
    
    sfreq = data.get('sfreq') # get sampling frequency

    # Ensure PPG signal and sampling rate are available
    if data is not None and sfreq is not None:

        bids_base_fname = data.get('bids_base_filename') # get base filename in BIDS format for current participant

        # Check whether at least one optional pre-processing step between normalization, 
        # filtering and cleaning has been carried out:
        if ppg_normalize or ppg_filter or clip_artifacts_correct: 

            # Step 6a: Get the raw and cleaned PPG Signal
            ppg_raw = data.get('ppg_raw')  # raw PPG data
            ppg_clean = data.get(
                'ppg_clipping_clean' if clip_artifacts_correct else 
                'ppg_filt' if ppg_filter else 'ppg_norm')

            # Save raw and cleaned PPG data to TSV.GZ file
            save_ppg_cleaned(ppg_raw, ppg_clean, subj_id, bids_base_fname)

        # Step 6b: Save PPG preprocessing information to JSON
        if 'ppg_peaks_idx' in data:
            ppg_preproc_dict = save_ppg_preproc(ppg_dict[subj_id], subj_id, bids_base_fname, manual_correct=manual_correct)
        
        else:
            # If R-peaks data is not present, skip further processing for this participant
            print(f'No PPG systolic peaks detected for participant {subj_id}. Skipping.')
            continue

        # Step 6c: Save interpolated Heart Rate Data (optional)
        if hr_interpol:
            save_hr_interpol(ppg_preproc_dict, sfreq, subj_id, bids_base_fname, interpol_type='cubic')

        print(f"----- Data processing completed for participant {subj_id} -----\n")

    else:
        print(f"Missing data or sampling rate for participant {subj_id}. Skipping.")

## **Good job, your PPG preprocessing is done!**

When using or adapting this BBSIG pipeline to conduct PPG preprocessing in your research work, please cite us in your publication as follows: 

**APA**

*Gerosa M., Agrawal N., Ciston A.B., Fischer A., Fourcade A., Koushik A., Neubauer M., Patyczek A., Piejka A., Reinwarth E., Roellecke L., Shum Y.H., Verschooren S., Gaebler M. (2025). Brain-Body Analysis Special Interest Group (BBSIG) (Version 0.0.1) [Computer software]. https://doi.org/10.5281/zenodo.15212797*